In [1]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import get_linear_schedule_with_warmup
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from accelerate import Accelerator
from conlleval import evaluate as conlleval_evaluate
from tqdm.auto import tqdm
import ast
import warnings
import gc
warnings.filterwarnings('ignore')

torch.cuda.empty_cache()
gc.collect()

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Capacity: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Capacity: 6.4 GB


In [2]:
df = pd.read_csv(r"data\ner_dataset.csv")
print(f"Loaded: {len(df)} строк")
df.head()

Loaded: 629538 строк


,token,tag,language,sentence_id,split
0,EU,B-ORG,en,en_train_0,train
1,rejects,O,en,en_train_0,train
2,German,B-MISC,en,en_train_0,train
3,call,O,en,en_train_0,train
4,to,O,en,en_train_0,train


In [3]:
need = {"token", "tag", "language", "sentence_id", "split"}
assert not (need - set(df.columns)), "No more columns"
print("Checked!")

Checked!


In [4]:
def to_sentence_level(df_split):
    g = df_split.groupby("sentence_id", sort=False)
    return pd.DataFrame({
        "sentence_id": g.size().index,
        "tokens": g["token"].apply(list).values,
        "ner_tags": g["tag"].apply(list).values,
        "language": g["language"].first().values,
        "split": g["split"].first().values,
    })

sent_df = to_sentence_level(df)
print(f"Sentences: {len(sent_df)}")

Sentences: 42507


In [5]:
def ensure_list(x):
    if isinstance(x, list): return x
    if pd.isna(x): return None
    if isinstance(x, str):
        x = x.strip()
        if x.startswith("[") and x.endswith("]"):
            try: return ast.literal_eval(x)
            except: return None
        return [x]
    return None

sent_df["tokens"] = sent_df["tokens"].apply(ensure_list)
sent_df["ner_tags"] = sent_df["ner_tags"].apply(ensure_list)
sent_df = sent_df.dropna(subset=["tokens", "ner_tags"]).reset_index(drop=True)
sent_df = sent_df[sent_df["tokens"].str.len() == sent_df["ner_tags"].str.len()].reset_index(drop=True)
print(f"После очистки: {len(sent_df)}")

После очистки: 42507


In [6]:
all_labels = sorted({tag for tags in sent_df["ner_tags"] for tag in tags})
if "O" in all_labels:
    all_labels = ["O"] + [x for x in all_labels if x != "O"]

label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}
print(f"Меток: {len(all_labels)}")

Меток: 9


In [7]:
train_df = sent_df[sent_df["split"] == "train"][["tokens", "ner_tags"]].reset_index(drop=True)
val_df = sent_df[sent_df["split"] == "validation"][["tokens", "ner_tags"]].reset_index(drop=True)
test_df = sent_df[sent_df["split"] == "test"][["tokens", "ner_tags"]].reset_index(drop=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Train: 34,005 | Val: 4,251 | Test: 4,251


In [8]:
MODEL_CKPT = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
print("Tokenizer is ready!")

Tokenizer is ready!


In [9]:
class OptimizedNERDataset(Dataset):
    def __init__(self, df_sent, tokenizer, label2id, max_length=128):  
        self.samples = []
        print(f"Tokenization of {len(df_sent)} samples...")
        
        for idx in tqdm(range(len(df_sent))):
            tokens = df_sent.iloc[idx]["tokens"]
            tags = df_sent.iloc[idx]["ner_tags"]
            
            if not isinstance(tokens, list) or len(tokens) == 0:
                continue
            
            tokens = [str(t) for t in tokens]
            
            enc = tokenizer(
                tokens,
                is_split_into_words=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )
            
            word_ids = enc.word_ids(batch_index=0)
            labels = []
            prev = None
            for wid in word_ids:
                if wid is None:
                    labels.append(-100)
                elif wid != prev:
                    labels.append(label2id[tags[wid]])
                else:
                    labels.append(-100)
                prev = wid
            
            self.samples.append({
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(labels, dtype=torch.long)
            })
        
        print(f"Done: {len(self.samples)} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

In [10]:
def collate_fn(batch):
    input_ids = torch.nn.utils.rnn.pad_sequence(
        [x["input_ids"] for x in batch], batch_first=True, padding_value=tokenizer.pad_token_id
    )
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        [x["attention_mask"] for x in batch], batch_first=True, padding_value=0
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        [x["labels"] for x in batch], batch_first=True, padding_value=-100
    )
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [11]:
train_ds = OptimizedNERDataset(train_df, tokenizer, label2id, max_length=128)

Tokenization of 34005 samples...


  0%|          | 0/34005 [00:00<?, ?it/s]

Done: 34005 samples


In [12]:
val_ds = OptimizedNERDataset(val_df, tokenizer, label2id, max_length=128)

Tokenization of 4251 samples...


  0%|          | 0/4251 [00:00<?, ?it/s]

Done: 4251 samples


In [13]:
test_ds = OptimizedNERDataset(test_df, tokenizer, label2id, max_length=128)

Tokenization of 4251 samples...


  0%|          | 0/4251 [00:00<?, ?it/s]

Done: 4251 samples


In [14]:
BATCH_SIZE = 4           
GRADIENT_ACCUMULATION = 8  
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
EVAL_STEPS = 1000        

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, 
    collate_fn=collate_fn, num_workers=0, pin_memory=False  
)

val_loader = DataLoader(
    val_ds, batch_size=8, shuffle=False,  
    collate_fn=collate_fn, num_workers=0, pin_memory=False
)

test_loader = DataLoader(
    test_ds, batch_size=8, shuffle=False,
    collate_fn=collate_fn, num_workers=0, pin_memory=False
)

print(f"DataLoaders created")
print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION}")
print(f"Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")

DataLoaders created
Batch size: 4
Gradient accumulation: 8
Effective batch: 32


In [15]:
print("First batch testing...")
test_batch = next(iter(train_loader))
print(f"OK | Input: {test_batch['input_ids'].shape}")
del test_batch
torch.cuda.empty_cache()

First batch testing...
OK | Input: torch.Size([4, 32])


In [16]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(all_labels),
    id2label=id2label,
    label2id=label2id
)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: 277,459,977 parameters


In [17]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
num_training_steps = NUM_EPOCHS * len(train_loader) // GRADIENT_ACCUMULATION
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)
print(f"Optimizer | steps: {num_training_steps}")

Optimizer | steps: 5313


In [18]:
accelerator = Accelerator(
    mixed_precision="fp16",  
    gradient_accumulation_steps=GRADIENT_ACCUMULATION
)

model, optimizer, train_loader, val_loader, test_loader, scheduler = accelerator.prepare(
    model, optimizer, train_loader, val_loader, test_loader, scheduler
)
print(f"Accelerator on {accelerator.device}")

Accelerator on cuda


In [19]:
def conlleval_metrics(y_true, y_pred):
    lines = []
    for true_seq, pred_seq in zip(y_true, y_pred):
        for t, p in zip(true_seq, pred_seq):
            lines.append(f"X {t} {p}")
        lines.append("")  

    res = conlleval_evaluate(lines)

    evals = res["overall"]["chunks"]["evals"]
    p = float(evals["prec"])
    r = float(evals["rec"])
    f = float(evals["f1"])

    return p, r, f

def run_eval(dataloader, max_batches=None):
    model.eval()
    all_preds, all_labels_true = [], []
    
    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if max_batches and i >= max_batches:
                break
            out = model(**batch)
            preds = accelerator.gather(out.logits.argmax(-1)).detach().cpu().numpy()
            labels = accelerator.gather(batch["labels"]).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels_true.extend(labels)
    
    return all_preds, all_labels_true

def decode_seqeval(preds, labels):
    y_true, y_pred = [], []
    for p_seq, l_seq in zip(preds, labels):
        sp, sl = [], []
        for p, l in zip(p_seq, l_seq):
            if l == -100: continue
            sp.append(id2label[int(p)])
            sl.append(id2label[int(l)])
        y_pred.append(sp)
        y_true.append(sl)
    return y_true, y_pred

In [20]:
print(f"\n{'='*60}")
print(f"{'='*60}")
print(f"Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Steps for epoch: {len(train_loader)}")
print(f"Total steps: {num_training_steps}")
print(f"{'='*60}\n")

global_step = 0
best_f1 = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    
    for batch_idx, batch in enumerate(progress_bar):
        with accelerator.accumulate(model):
            out = model(**batch)
            loss = out.loss
            accelerator.backward(loss)
            
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            total_loss += loss.item()
            
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "avg": f"{total_loss/(batch_idx+1):.4f}"
            })
            
            global_step += 1
            
            if global_step % EVAL_STEPS == 0:
                torch.cuda.empty_cache()  
                val_preds, val_labels = run_eval(val_loader, max_batches=50)
                y_true, y_pred = decode_seqeval(val_preds, val_labels)
                p, r, f = conlleval_metrics(y_true, y_pred)
                print(f"\nStep {global_step} | F1={f:.4f}")
                model.train()
                torch.cuda.empty_cache()
    
    avg_loss = total_loss / len(train_loader)
    torch.cuda.empty_cache()
    val_preds, val_labels = run_eval(val_loader)
    y_true, y_pred = decode_seqeval(val_preds, val_labels)
    p, r, f = conlleval_metrics(y_true, y_pred)
    
    print(f"\n{'='*60}")
    print(f"Epoch {epoch} | Loss={avg_loss:.4f} | P={p:.4f} R={r:.4f} F1={f:.4f}")
    print(f"{'='*60}\n")
    
    if f > best_f1:
        best_f1 = f
        print(f"Best model! F1={best_f1:.4f}\n")
        accelerator.wait_for_everyone()
        unwrapped_model = accelerator.unwrap_model(model)
        unwrapped_model.save_pretrained("best_ml_model")
        tokenizer.save_pretrained("best_ml_model")
    
    torch.cuda.empty_cache()
    gc.collect()

print(f"\nDone! Best F1: {best_f1:.4f}")


Effective batch: 32
Steps for epoch: 8502
Total steps: 5313



Epoch 1/5:   0%|          | 0/8502 [00:00<?, ?it/s]


Step 1000 | F1=0.0000

Step 2000 | F1=0.2592

Step 3000 | F1=0.6879

Step 4000 | F1=0.8439

Step 5000 | F1=0.8817

Step 6000 | F1=0.8992

Step 7000 | F1=0.9043

Step 8000 | F1=0.9042

Epoch 1 | Loss=0.3033 | P=0.8662 R=0.9045 F1=0.8849

Best model! F1=0.8849



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/5:   0%|          | 0/8502 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5/5:   0%|          | 0/8502 [00:00<?, ?it/s]


Step 35000 | F1=0.9601

Step 36000 | F1=0.9562

Step 37000 | F1=0.9577

Step 38000 | F1=0.9537

Step 39000 | F1=0.9562

Step 40000 | F1=0.9512

Step 41000 | F1=0.9577

Step 42000 | F1=0.9545

Epoch 5 | Loss=0.0170 | P=0.9442 R=0.9516 F1=0.9479

Best model! F1=0.9479



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Done! Best F1: 0.9479


In [21]:
print("Testing...")
torch.cuda.empty_cache()
test_preds, test_labels = run_eval(test_loader)
y_true_t, y_pred_t = decode_seqeval(test_preds, test_labels)
p, r, f = conlleval_metrics(y_true_t, y_pred_t)

print(f"\n{'='*60}")
print(f"TEST | Precision={p:.4f} Recall={r:.4f} F1={f:.4f}")
print(f"{'='*60}")

Testing...

TEST | Precision=0.9391 Recall=0.9468 F1=0.9429
